In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

tr = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
train = tr.copy()
test_1 = test.copy()
print(train.info())
print(train.describe(include="all"))
print(test.head(5))
y= train["Survived"]
x = train.drop(columns=["Survived"])

#missing Values handling

x["Age"]= x["Age"].fillna(x["Age"].median())
test["Age"] = test["Age"].fillna(test["Age"].median())
x.info()
x.head()
x.describe(include="all")

x["Embarked"] = x["Embarked"].fillna(x["Embarked"].mode()[0])
test["Embarked"] = test["Embarked"].fillna(x["Embarked"].mode()[0])

fare_median = x["Fare"].median()

x["Fare"] = x["Fare"].fillna(fare_median)
test["Fare"] = test["Fare"].fillna(fare_median)


x["FamilySize"] = x["SibSp"] +x["Parch"]+ 1
test["FamilySize"] = test["SibSp"] +test["Parch"]+ 1

x["IsAlone"] = (x["FamilySize"]==1).astype(int)
test["IsAlone"] = (test["FamilySize"]==1).astype(int)


x["Sex"] = x["Sex"].map({"male": 0, "female": 1})
test["Sex"] = test["Sex"].map({"male": 0, "female": 1})

x = pd.get_dummies(x, columns=["Embarked"], drop_first=True)
test = pd.get_dummies(test, columns=["Embarked"], drop_first=True)
x, test = x.align(test, join="left", axis=1, fill_value=0)

x = x.drop(columns=["PassengerId","Name", "Ticket", "Cabin"])
test = test.drop(columns=["PassengerId","Name", "Ticket", "Cabin"])

model = LogisticRegression(max_iter=1000)
model.fit(x, y)

scores = cross_val_score(model, x, y, cv=5, scoring="accuracy")
print("CV Score: " , scores)
print("mean accuracy: ", scores.mean())
coeffs = pd.Series(model.coef_[0], index=x.columns).sort_values()
coeffs
test_preds = model.predict(test)
tit_submission = pd.DataFrame({
    "PassengerId": test_1["PassengerId"],
    "Survived": test_preds
})

tit_submission.to_csv("submission.csv", index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Pclass       891 non-null    int64  
 2   Name         891 non-null    object 
 3   Sex          891 non-null    object 
 4   Age          891 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Ticket       891 non-null    object 
 8   Fare         891 non-null    float64
 9   Cabin        204 non-null    object 
 10  Embarked     889 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 76.7+ KB
CV Score:  [0.78212291 0.76966292 0.79775281 0.78089888 0.83146067]
mean accuracy:  0.792379637185362
